# Other Utilities — Advanced Reference

| Utility | Purpose |
|---|---|
| `.cols(ascending=True/False/None)` | List column names sorted or in original order |
| `.handle_missing(fillna='.')` | Fill string NaN with a marker, numeric NaN with 0, strip whitespace |
| `.group_x(group, aggfunc, value, dropna)` | Add a group-level scalar back to every row (via `transform`) |
| `.clip()` | Copy DataFrame to clipboard (tab-separated, no index) |

---

In [ ]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']

## `cols()` — use to drive `select` dynamically
`cols()` returns a plain list — combine it with list comprehensions inside `select` for programmatic column selection.

In [ ]:
# Alphabetical column list — useful for quick discovery
penguins.cols()

In [ ]:
# Descending alphabetical sort — useful when column names form a natural reverse order
penguins.cols(ascending=False)

In [ ]:
# Use cols() to select only columns containing 'mm', preserving original order
(penguins
 .select([c for c in penguins.cols(ascending=None) if 'mm' in c])
 .head()
)

## `handle_missing()` — clean before aggregating
Fills string/category NaN with a visible marker (default `.`) and numeric NaN with `0`. Also strips leading/trailing whitespace from string columns.

In [ ]:
# Default fillna='.': NaN sex becomes '.', NaN numerics become 0
# Then aggregate to see the '.' group alongside real groups
(penguins
 .handle_missing()
 .select(['species', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
)

In [ ]:
# Custom fillna marker — makes missing group explicit in reports
(penguins
 .handle_missing(fillna='Unknown')
 .select(['species', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
 .qry({'sex': 'Unknown'})  # spot-check the imputed group
)

In [ ]:
# Use a single-char marker like '$' to make missing data stand out in reports
(penguins
 .handle_missing(fillna='$')
 .select(['species', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
)

## `group_x()` — broadcast a group aggregate back to every row
Unlike `agg_df` which collapses rows, `group_x` **adds a column** to the original DataFrame using `transform`. Useful for computing a group value alongside row-level detail (e.g., share of group total, deviation from group mean).

In [ ]:
# Default: count per group (aggfunc='n') — adds column 'n' to every row
(penguins
 .group_x(group=['species', 'island'])
 .select(['species', 'island', 'sex', 'body_mass_g', 'n'])
 .head(10)
)

In [ ]:
# group_x with value + aggfunc: add group mean of body_mass_g to every row
# Then compute each penguin's deviation from its species group mean
(penguins
 .group_x(group=['species'], value='body_mass_g', aggfunc='mean')
 .assign(deviation=lambda df: df['body_mass_g'] - df['x'])
 .select(['species', 'island', 'sex', 'body_mass_g', 'x', 'deviation'])
 .rename(columns={'x': 'species_mean_mass'})
 .head(10)
)

In [ ]:
# group_x + dropna=False: also compute group counts for NaN groups
(penguins
 .group_x(group=['species', 'sex'], dropna=False)
 .select(['species', 'sex', 'n'])
 .agg_df(aggfunc='max')   # one row per group showing the group size
)

In [ ]:
# .pipe(pt.group_x, ...) syntax — identical result, preferred when chaining with pandas methods
# that don't support the method-on-DataFrame syntax directly
(penguins
 .pipe(pt.group_x, group=['species', 'sex'], value='body_mass_g', aggfunc='mean')
 .assign(deviation=lambda df: df['body_mass_g'] - df['x'])
 .select(['species', 'sex', 'body_mass_g', 'x', 'deviation'])
 .rename(columns={'x': 'group_mean_mass'})
 .head(10)
)

## `clip()` — copy to clipboard
`clip()` copies the DataFrame to the system clipboard as tab-separated text (no index). Paste directly into Excel, Google Sheets, or any text editor. Invisible in notebook output — run it and then paste.

In [ ]:
# clip() copies the current DataFrame to clipboard — paste into Excel/Sheets
# Run this cell then Cmd+V in any spreadsheet application
(penguins
 .select(['species', 'island', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
 .clip()
)